# **SET UP LIBRARY DAN PATH**

In [1]:
# =========================
# 0-1) Init + Setup path + Load metadata (STRICT) + Exclude VAD drop
# =========================

import os, sys, re, json, math, random, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Torch optional (kalau nanti SSL)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except Exception:
    torch = None
    DEVICE = "cpu"

def pick_existing(paths: List[Path]) -> Path:
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError("Tidak ada path yang ditemukan:\n" + "\n".join(map(str, paths)))

# --- auto-detect ROOT (cari output/split_strict/manifest_strict.csv)
CWD = Path.cwd().resolve()
ROOT = None
for p in [CWD] + list(CWD.parents):
    if (p / "output" / "split_strict" / "manifest_strict.csv").exists():
        ROOT = p
        break
    if (p / "split_strict" / "manifest_strict.csv").exists():  # fallback kalau bukan di output/
        ROOT = p
        break

if ROOT is None:
    raise FileNotFoundError(
        "Gagal nemu ROOT. Pastikan ada 'output/split_strict/manifest_strict.csv' "
        "di salah satu parent folder dari notebook kamu."
    )

# --- strict split dir + manifest
STRICT_DIR = pick_existing([
    ROOT / "output" / "split_strict",
    ROOT / "split_strict",
])
MANIFEST_STRICT = STRICT_DIR / "manifest_strict.csv"

# --- audio hasil preprocessing (default seperti baseline official)
AUDIO_DIR = None
for c in [
    ROOT / "output" / "preprocessing" / "preprocessed_full",
    ROOT / "preprocessing" / "preprocessed_full",
]:
    if c.exists():
        AUDIO_DIR = c
        break

# --- output root untuk baseline strict (bedain dari baseline_official)
OUT_ROOT = ROOT / "output" / "baseline_strict"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# --- VAD drop path
VAD_DROP = ROOT / "output" / "preprocessing" / "vad" / "vad_drop.csv"
if not VAD_DROP.exists():
    raise FileNotFoundError(f"vad_drop.csv tidak ditemukan di: {VAD_DROP}")

def exclude_vad_drop(df: pd.DataFrame, vad_drop_path: Path) -> pd.DataFrame:
    drop_df = pd.read_csv(vad_drop_path)
    if "clip_id" not in drop_df.columns:
        raise ValueError(f"vad_drop.csv harus punya kolom 'clip_id'. Kolom tersedia: {drop_df.columns.tolist()}")
    initial_len = len(df)
    df = df[~df["clip_id"].isin(drop_df["clip_id"].astype(str).values)].reset_index(drop=True)
    final_len = len(df)
    print(f"Excluded {initial_len - final_len} samples due to VAD drop.")
    print(f"New shape: {df.shape}")
    print(f"Size before drop: {initial_len}, size after drop: {final_len}\n")
    return df

# --- load strict manifest
df_all = pd.read_csv(MANIFEST_STRICT)
if "clip_id" not in df_all.columns:
    raise ValueError(f"Kolom 'clip_id' tidak ada. Kolom tersedia: {df_all.columns.tolist()}")
df_all["clip_id"] = df_all["clip_id"].astype(str)

# normalisasi nama kolom split
if "split_strict" not in df_all.columns:
    if "split" in df_all.columns:
        df_all = df_all.rename(columns={"split": "split_strict"})
    elif "split_strict_x" in df_all.columns:
        df_all = df_all.rename(columns={"split_strict_x": "split_strict"})
    else:
        raise ValueError(f"Kolom split strict tidak ditemukan. Kolom tersedia: {df_all.columns.tolist()}")

# --- buat df_train/df_val/df_test (strict)
spl = df_all["split_strict"].astype(str).str.lower()
df_train = df_all[spl == "train"].reset_index(drop=True)
df_val   = df_all[spl == "val"].reset_index(drop=True)
df_test  = df_all[spl == "test"].reset_index(drop=True)

# --- exclude VAD drop per split (lebih jelas log-nya)
print("=== VAD DROP FILTER ===")
print("[train]")
df_train = exclude_vad_drop(df_train, VAD_DROP)
print("[val]")
df_val   = exclude_vad_drop(df_val, VAD_DROP)
print("[test]")
df_test  = exclude_vad_drop(df_test, VAD_DROP)

# --- helper buat resolve path wav (kalau mau pakai audio_out dari manifest_strict)
def resolve_wav_path(clip_id: str, audio_out: Optional[str] = None) -> Path:
    # 1) kalau manifest punya audio_out dan itu valid
    if audio_out is not None and str(audio_out).strip() != "" and str(audio_out).lower() != "nan":
        s = str(audio_out).replace("\\", "/")
        p = Path(s)
        if p.is_absolute() and p.exists():
            return p
        if s.startswith("output/"):
            cand = ROOT / s
            if cand.exists():
                return cand

    # 2) fallback: pakai folder preprocessed_full seperti notebook baseline official
    if AUDIO_DIR is not None:
        cand = AUDIO_DIR / f"{clip_id}.wav"
        if cand.exists():
            return cand

    raise FileNotFoundError(f"WAV tidak ketemu untuk clip_id={clip_id}. Cek kolom audio_out / AUDIO_DIR.")

print("Init OK (STRICT + VAD DROP)")
print("ROOT        :", ROOT)
print("STRICT_DIR  :", STRICT_DIR)
print("MANIFEST    :", MANIFEST_STRICT)
print("VAD_DROP    :", VAD_DROP)
print("AUDIO_DIR   :", AUDIO_DIR if AUDIO_DIR else "(None, pakai audio_out)")
print("OUT_ROOT    :", OUT_ROOT)
print("Split shapes:", "train", df_train.shape, "| val", df_val.shape, "| test", df_test.shape)
if "group_id" in df_all.columns:
    g = pd.concat([df_train.assign(_s="train"), df_val.assign(_s="val"), df_test.assign(_s="test")]) \
          .groupby("_s")["group_id"].nunique().to_dict()
    print("Unique group_id:", {k: int(v) for k, v in g.items()})
print("SEED  :", SEED)
print("DEVICE:", DEVICE)
print("pandas:", pd.__version__, "| numpy:", np.__version__)


c:\Users\aquq1\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== VAD DROP FILTER ===
[train]
Excluded 0 samples due to VAD drop.
New shape: (5936, 14)
Size before drop: 5936, size after drop: 5936

[val]
Excluded 0 samples due to VAD drop.
New shape: (1999, 14)
Size before drop: 1999, size after drop: 1999

[test]
Excluded 0 samples due to VAD drop.
New shape: (2039, 14)
Size before drop: 2039, size after drop: 2039

Init OK (STRICT + VAD DROP)
ROOT        : E:\tugas-akhir-qiqi
STRICT_DIR  : E:\tugas-akhir-qiqi\output\split_strict
MANIFEST    : E:\tugas-akhir-qiqi\output\split_strict\manifest_strict.csv
VAD_DROP    : E:\tugas-akhir-qiqi\output\preprocessing\vad\vad_drop.csv
AUDIO_DIR   : E:\tugas-akhir-qiqi\output\preprocessing\preprocessed_full
OUT_ROOT    : E:\tugas-akhir-qiqi\output\baseline_strict
Split shapes: train (5936, 14) | val (1999, 14) | test (2039, 14)
Unique group_id: {'test': 617, 'train': 1829, 'val': 608}
SEED  : 42
DEVICE: cpu
pandas: 2.2.1 | numpy: 1.26.4


# **REUSE EMBEDDING AND ALIGNING TO STRICT - HUBERT**

In [14]:
# =========================
# 2) Reuse embedding dari baseline_official -> align ke STRICT split (by clip_id)
# =========================

import numpy as np
import pandas as pd
from pathlib import Path

assert "ROOT" in globals()
assert "df_train" in globals() and "df_val" in globals() and "df_test" in globals()

BACKBONE_KEY = "hubert"   # ganti: "wav2vec2" / "wavlm"

# lokasi embedding yang sudah ada (official)
EMB_DIR_OFF = ROOT / "output" / "baseline_official" / "embeddings" / BACKBONE_KEY
assert EMB_DIR_OFF.exists(), f"Folder embedding official tidak ada: {EMB_DIR_OFF}"

def load_official_split(split: str):
    X = np.load(EMB_DIR_OFF / f"{split}_emb.npy", mmap_mode="r")
    ids = pd.read_csv(EMB_DIR_OFF / f"{split}_clip_id.csv")["clip_id"].astype(str).tolist()
    return X, ids

# load pool embedding dari official train/val/test
X_off = {}
ids_off = {}

for split in ["train", "val", "test"]:
    X_off[split], ids_off[split] = load_official_split(split)
    print(f"[OFFICIAL] {split}: X={X_off[split].shape} ids={len(ids_off[split])}")

# build mapping clip_id -> (split, idx)
id2loc = {}
dup = 0
for split in ["train", "val", "test"]:
    for i, cid in enumerate(ids_off[split]):
        if cid in id2loc:
            dup += 1
        id2loc[cid] = (split, i)

if dup > 0:
    print(f"[WARN] Duplicated clip_id across official splits: {dup} (harusnya 0).")

H = int(X_off["train"].shape[1])
print("Hidden size (H):", H)

def build_X_from_strict(df_split: pd.DataFrame, name: str):
    ids = df_split["clip_id"].astype(str).tolist()
    X = np.zeros((len(ids), H), dtype=np.float32)

    missing = []
    for j, cid in enumerate(ids):
        loc = id2loc.get(cid)
        if loc is None:
            missing.append(cid)
            continue
        sp, i = loc
        X[j] = X_off[sp][i]  # ambil dari memmap official

    if missing:
        print(f"[WARN] {name}: missing embeddings = {len(missing)} / {len(ids)} (contoh 5): {missing[:5]}")
        # kalau kamu mau strict fail biar ketahuan:
        # raise ValueError(f"Missing embeddings in {name}: {len(missing)}")

    print(f"[STRICT] {name}: X={X.shape} ids={len(ids)}")
    return X, ids

# align ke strict split
X_train, train_ids = build_X_from_strict(df_train, "train")
X_val,   val_ids   = build_X_from_strict(df_val,   "val")
X_test,  test_ids  = build_X_from_strict(df_test,  "test")

# (opsional) cache ulang supaya notebook strict mirip baseline official
STRICT_EMB_DIR = ROOT / "output" / "baseline_strict" / "embeddings" / BACKBONE_KEY
STRICT_EMB_DIR.mkdir(parents=True, exist_ok=True)

np.save(STRICT_EMB_DIR / "train_emb.npy", X_train)
np.save(STRICT_EMB_DIR / "val_emb.npy",   X_val)
np.save(STRICT_EMB_DIR / "test_emb.npy",  X_test)

pd.DataFrame({"clip_id": train_ids}).to_csv(STRICT_EMB_DIR / "train_clip_id.csv", index=False)
pd.DataFrame({"clip_id": val_ids}).to_csv(STRICT_EMB_DIR / "val_clip_id.csv", index=False)
pd.DataFrame({"clip_id": test_ids}).to_csv(STRICT_EMB_DIR / "test_clip_id.csv", index=False)

print("Saved strict-aligned embeddings to:", STRICT_EMB_DIR)


[OFFICIAL] train: X=(5988, 768) ids=5988
[OFFICIAL] val: X=(1994, 768) ids=1994
[OFFICIAL] test: X=(1992, 768) ids=1992
Hidden size (H): 768
[STRICT] train: X=(5936, 768) ids=5936
[STRICT] val: X=(1999, 768) ids=1999
[STRICT] test: X=(2039, 768) ids=2039
Saved strict-aligned embeddings to: E:\tugas-akhir-qiqi\output\baseline_strict\embeddings\hubert


In [15]:
# =========================
# 3) Train Ridge Regressor (baseline) + Eval (STRICT)
# =========================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# --- pilih backbone yang embedding-nya kamu reuse
BACKBONE_KEY = "hubert"  # "wav2vec2" / "wavlm"

assert "ROOT" in globals()
assert "df_train" in globals() and "df_val" in globals() and "df_test" in globals()

EMB_DIR = ROOT / "output" / "baseline_strict" / "embeddings" / BACKBONE_KEY
assert EMB_DIR.exists(), f"EMB_DIR strict tidak ada: {EMB_DIR}"

def load_split(split: str):
    X = np.load(EMB_DIR / f"{split}_emb.npy")
    ids = pd.read_csv(EMB_DIR / f"{split}_clip_id.csv")["clip_id"].astype(str).tolist()
    return X, ids

def align_X_to_df(X, ids, df, name=""):
    id2i = {cid: i for i, cid in enumerate(ids)}
    target_ids = df["clip_id"].astype(str).tolist()

    order = []
    missing = []
    for cid in target_ids:
        i = id2i.get(cid)
        if i is None:
            missing.append(cid)
            order.append(-1)
        else:
            order.append(i)

    if missing:
        print(f"[WARN] {name}: missing embeddings = {len(missing)} / {len(target_ids)} (contoh 5): {missing[:5]}")
        # kalau mau strict fail:
        raise ValueError(f"Missing embeddings for {name}: {len(missing)}")

    X_aligned = X[np.array(order)]
    return X_aligned, target_ids

# --- load + align to strict dfs (important)
X_train_raw, train_ids_raw = load_split("train")
X_val_raw,   val_ids_raw   = load_split("val")
X_test_raw,  test_ids_raw  = load_split("test")

X_train, train_ids = align_X_to_df(X_train_raw, train_ids_raw, df_train, "train")
X_val,   val_ids   = align_X_to_df(X_val_raw,   val_ids_raw,   df_val,   "val")
X_test,  test_ids  = align_X_to_df(X_test_raw,  test_ids_raw,  df_test,  "test")

print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

# --- auto-detect label columns (ambil 5 kolom numerik yang bukan id/path/group/split)
def detect_label_cols(df):
    exclude = set([c for c in df.columns if any(k in c.lower() for k in ["clip", "id", "path", "file", "group", "split", "ethnicity", "gender", "avg_trait"])])
    num_cols = [c for c in df.columns if (c not in exclude) and pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) < 5:
        raise ValueError(f"Kolom numerik kandidat label kurang dari 5: {num_cols}")
    return num_cols[:5]

label_cols = detect_label_cols(df_train)
print("Label cols:", label_cols)

y_train = df_train[label_cols].to_numpy(dtype=np.float32)
y_val   = df_val[label_cols].to_numpy(dtype=np.float32)
y_test  = df_test[label_cols].to_numpy(dtype=np.float32)

# --- scale embeddings (fit hanya di train)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

def metrics(y_true, y_pred, name=""):
    mae  = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))
    r2   = r2_score(y_true, y_pred, multioutput="raw_values")

    acc = 1.0 - mae
    mean_acc = acc.mean()

    dfm = pd.DataFrame({
        "trait": label_cols,
        "Acc(1-MAE)": acc,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })
    dfm.loc["mean"] = ["mean", mean_acc, mae.mean(), rmse.mean(), r2.mean()]

    print(f"\n== {name} ==")
    try:
        display(dfm)
    except Exception:
        print(dfm)
    return dfm

# --- (opsional) tuning alpha pakai mean MAE val
ALPHAS = [0.1, 1.0, 10.0, 100.0]
best_alpha = None
best_val = 1e9
best_model = None

for a in ALPHAS:
    ridge = Ridge(alpha=a, random_state=42)
    model = MultiOutputRegressor(ridge)
    model.fit(X_train_s, y_train)
    pred_val = model.predict(X_val_s)
    mae_val = mean_absolute_error(y_val, pred_val)  # mean MAE (gabungan 5 trait)
    print(f"alpha={a:<6} | val mean MAE={mae_val:.6f}")
    if mae_val < best_val:
        best_val = mae_val
        best_alpha = a
        best_model = model

print(f"\nBest alpha = {best_alpha} (val mean MAE = {best_val:.6f})")

# --- final predict (pakai best alpha)
model = best_model
pred_val  = model.predict(X_val_s)
pred_test = model.predict(X_test_s)

m_val  = metrics(y_val,  pred_val,  f"{BACKBONE_KEY} | STRICT | VAL")
m_test = metrics(y_test, pred_test, f"{BACKBONE_KEY} | STRICT | TEST")

# --- save outputs
OUT_DIR = ROOT / "output" / "baseline_strict" / "results" / BACKBONE_KEY
OUT_DIR.mkdir(parents=True, exist_ok=True)

# save preds
pd.DataFrame({"clip_id": val_ids,  **{f"pred_{c}": pred_val[:,i]  for i,c in enumerate(label_cols)}})\
  .to_csv(OUT_DIR / "pred_val.csv", index=False)
pd.DataFrame({"clip_id": test_ids, **{f"pred_{c}": pred_test[:,i] for i,c in enumerate(label_cols)}})\
  .to_csv(OUT_DIR / "pred_test.csv", index=False)

# save metrics
m_val.to_csv(OUT_DIR / "metrics_val.csv", index=False)
m_test.to_csv(OUT_DIR / "metrics_test.csv", index=False)

# save scaler + model
joblib.dump(scaler, OUT_DIR / "scaler.joblib")
joblib.dump(model,  OUT_DIR / "ridge_multioutput.joblib")

# save meta
meta = {
    "backbone": BACKBONE_KEY,
    "alpha": float(best_alpha),
    "label_cols": label_cols,
    "shapes": {"train": list(X_train.shape), "val": list(X_val.shape), "test": list(X_test.shape)},
}
import json
(Path(OUT_DIR) / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("\nSaved to:", OUT_DIR)


X_train: (5936, 768) | X_val: (1999, 768) | X_test: (2039, 768)
Label cols: ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
alpha=0.1    | val mean MAE=0.104011
alpha=1.0    | val mean MAE=0.103910
alpha=10.0   | val mean MAE=0.103158
alpha=100.0  | val mean MAE=0.101027

Best alpha = 100.0 (val mean MAE = 0.101027)

== hubert | STRICT | VAL ==


,trait,Acc(1-MAE),MAE,RMSE,R2
0,extraversion,0.898492,0.101508,0.126747,0.277348
1,neuroticism,0.897575,0.102425,0.128923,0.262564
2,agreeableness,0.902024,0.097976,0.124112,0.114129
3,conscientiousness,0.896416,0.103584,0.131440,0.246699
4,openness,0.900356,0.099644,0.125336,0.243692
mean,mean,0.898973,0.101027,0.127312,0.228886



== hubert | STRICT | TEST ==


,trait,Acc(1-MAE),MAE,RMSE,R2
0,extraversion,0.896695,0.103305,0.129709,0.294159
1,neuroticism,0.894238,0.105762,0.133937,0.293466
2,agreeableness,0.899575,0.100425,0.127051,0.156178
3,conscientiousness,0.893857,0.106143,0.134082,0.289990
4,openness,0.898827,0.101173,0.126956,0.274042
mean,mean,0.896639,0.103362,0.130347,0.261567



Saved to: E:\tugas-akhir-qiqi\output\baseline_strict\results\hubert


# **REUSE EMBEDDING AND ALIGNING TO STRICT - WAV2VEC2**

In [16]:
# =========================
# 2) Reuse embedding dari baseline_official -> align ke STRICT split (by clip_id)
# =========================

import numpy as np
import pandas as pd
from pathlib import Path

assert "ROOT" in globals()
assert "df_train" in globals() and "df_val" in globals() and "df_test" in globals()

BACKBONE_KEY = "wav2vec2"   # ganti: "wav2vec2" / "wavlm"

# lokasi embedding yang sudah ada (official)
EMB_DIR_OFF = ROOT / "output" / "baseline_official" / "embeddings" / BACKBONE_KEY
assert EMB_DIR_OFF.exists(), f"Folder embedding official tidak ada: {EMB_DIR_OFF}"

def load_official_split(split: str):
    X = np.load(EMB_DIR_OFF / f"{split}_emb.npy", mmap_mode="r")
    ids = pd.read_csv(EMB_DIR_OFF / f"{split}_clip_id.csv")["clip_id"].astype(str).tolist()
    return X, ids

# load pool embedding dari official train/val/test
X_off = {}
ids_off = {}

for split in ["train", "val", "test"]:
    X_off[split], ids_off[split] = load_official_split(split)
    print(f"[OFFICIAL] {split}: X={X_off[split].shape} ids={len(ids_off[split])}")

# build mapping clip_id -> (split, idx)
id2loc = {}
dup = 0
for split in ["train", "val", "test"]:
    for i, cid in enumerate(ids_off[split]):
        if cid in id2loc:
            dup += 1
        id2loc[cid] = (split, i)

if dup > 0:
    print(f"[WARN] Duplicated clip_id across official splits: {dup} (harusnya 0).")

H = int(X_off["train"].shape[1])
print("Hidden size (H):", H)

def build_X_from_strict(df_split: pd.DataFrame, name: str):
    ids = df_split["clip_id"].astype(str).tolist()
    X = np.zeros((len(ids), H), dtype=np.float32)

    missing = []
    for j, cid in enumerate(ids):
        loc = id2loc.get(cid)
        if loc is None:
            missing.append(cid)
            continue
        sp, i = loc
        X[j] = X_off[sp][i]  # ambil dari memmap official

    if missing:
        print(f"[WARN] {name}: missing embeddings = {len(missing)} / {len(ids)} (contoh 5): {missing[:5]}")
        # kalau kamu mau strict fail biar ketahuan:
        # raise ValueError(f"Missing embeddings in {name}: {len(missing)}")

    print(f"[STRICT] {name}: X={X.shape} ids={len(ids)}")
    return X, ids

# align ke strict split
X_train, train_ids = build_X_from_strict(df_train, "train")
X_val,   val_ids   = build_X_from_strict(df_val,   "val")
X_test,  test_ids  = build_X_from_strict(df_test,  "test")

# (opsional) cache ulang supaya notebook strict mirip baseline official
STRICT_EMB_DIR = ROOT / "output" / "baseline_strict" / "embeddings" / BACKBONE_KEY
STRICT_EMB_DIR.mkdir(parents=True, exist_ok=True)

np.save(STRICT_EMB_DIR / "train_emb.npy", X_train)
np.save(STRICT_EMB_DIR / "val_emb.npy",   X_val)
np.save(STRICT_EMB_DIR / "test_emb.npy",  X_test)

pd.DataFrame({"clip_id": train_ids}).to_csv(STRICT_EMB_DIR / "train_clip_id.csv", index=False)
pd.DataFrame({"clip_id": val_ids}).to_csv(STRICT_EMB_DIR / "val_clip_id.csv", index=False)
pd.DataFrame({"clip_id": test_ids}).to_csv(STRICT_EMB_DIR / "test_clip_id.csv", index=False)

print("Saved strict-aligned embeddings to:", STRICT_EMB_DIR)


[OFFICIAL] train: X=(5988, 768) ids=5988
[OFFICIAL] val: X=(1994, 768) ids=1994
[OFFICIAL] test: X=(1992, 768) ids=1992
Hidden size (H): 768
[STRICT] train: X=(5936, 768) ids=5936
[STRICT] val: X=(1999, 768) ids=1999
[STRICT] test: X=(2039, 768) ids=2039
Saved strict-aligned embeddings to: E:\tugas-akhir-qiqi\output\baseline_strict\embeddings\wav2vec2


In [17]:
# =========================
# 3) Train Ridge Regressor (baseline) + Eval (STRICT)
# =========================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# --- pilih backbone yang embedding-nya kamu reuse
BACKBONE_KEY = "wav2vec2"  # "wav2vec2" / "wavlm"

assert "ROOT" in globals()
assert "df_train" in globals() and "df_val" in globals() and "df_test" in globals()

EMB_DIR = ROOT / "output" / "baseline_strict" / "embeddings" / BACKBONE_KEY
assert EMB_DIR.exists(), f"EMB_DIR strict tidak ada: {EMB_DIR}"

def load_split(split: str):
    X = np.load(EMB_DIR / f"{split}_emb.npy")
    ids = pd.read_csv(EMB_DIR / f"{split}_clip_id.csv")["clip_id"].astype(str).tolist()
    return X, ids

def align_X_to_df(X, ids, df, name=""):
    id2i = {cid: i for i, cid in enumerate(ids)}
    target_ids = df["clip_id"].astype(str).tolist()

    order = []
    missing = []
    for cid in target_ids:
        i = id2i.get(cid)
        if i is None:
            missing.append(cid)
            order.append(-1)
        else:
            order.append(i)

    if missing:
        print(f"[WARN] {name}: missing embeddings = {len(missing)} / {len(target_ids)} (contoh 5): {missing[:5]}")
        # kalau mau strict fail:
        raise ValueError(f"Missing embeddings for {name}: {len(missing)}")

    X_aligned = X[np.array(order)]
    return X_aligned, target_ids

# --- load + align to strict dfs (important)
X_train_raw, train_ids_raw = load_split("train")
X_val_raw,   val_ids_raw   = load_split("val")
X_test_raw,  test_ids_raw  = load_split("test")

X_train, train_ids = align_X_to_df(X_train_raw, train_ids_raw, df_train, "train")
X_val,   val_ids   = align_X_to_df(X_val_raw,   val_ids_raw,   df_val,   "val")
X_test,  test_ids  = align_X_to_df(X_test_raw,  test_ids_raw,  df_test,  "test")

print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

# --- auto-detect label columns (ambil 5 kolom numerik yang bukan id/path/group/split)
def detect_label_cols(df):
    exclude = set([c for c in df.columns if any(k in c.lower() for k in ["clip", "id", "path", "file", "group", "split", "ethnicity", "gender", "avg_trait"])])
    num_cols = [c for c in df.columns if (c not in exclude) and pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) < 5:
        raise ValueError(f"Kolom numerik kandidat label kurang dari 5: {num_cols}")
    return num_cols[:5]

label_cols = detect_label_cols(df_train)
print("Label cols:", label_cols)

y_train = df_train[label_cols].to_numpy(dtype=np.float32)
y_val   = df_val[label_cols].to_numpy(dtype=np.float32)
y_test  = df_test[label_cols].to_numpy(dtype=np.float32)

# --- scale embeddings (fit hanya di train)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

def metrics(y_true, y_pred, name=""):
    mae  = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))
    r2   = r2_score(y_true, y_pred, multioutput="raw_values")

    acc = 1.0 - mae
    mean_acc = acc.mean()

    dfm = pd.DataFrame({
        "trait": label_cols,
        "Acc(1-MAE)": acc,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })
    dfm.loc["mean"] = ["mean", mean_acc, mae.mean(), rmse.mean(), r2.mean()]

    print(f"\n== {name} ==")
    try:
        display(dfm)
    except Exception:
        print(dfm)
    return dfm

# --- (opsional) tuning alpha pakai mean MAE val
ALPHAS = [0.1, 1.0, 10.0, 100.0]
best_alpha = None
best_val = 1e9
best_model = None

for a in ALPHAS:
    ridge = Ridge(alpha=a, random_state=42)
    model = MultiOutputRegressor(ridge)
    model.fit(X_train_s, y_train)
    pred_val = model.predict(X_val_s)
    mae_val = mean_absolute_error(y_val, pred_val)  # mean MAE (gabungan 5 trait)
    print(f"alpha={a:<6} | val mean MAE={mae_val:.6f}")
    if mae_val < best_val:
        best_val = mae_val
        best_alpha = a
        best_model = model

print(f"\nBest alpha = {best_alpha} (val mean MAE = {best_val:.6f})")

# --- final predict (pakai best alpha)
model = best_model
pred_val  = model.predict(X_val_s)
pred_test = model.predict(X_test_s)

m_val  = metrics(y_val,  pred_val,  f"{BACKBONE_KEY} | STRICT | VAL")
m_test = metrics(y_test, pred_test, f"{BACKBONE_KEY} | STRICT | TEST")

# --- save outputs
OUT_DIR = ROOT / "output" / "baseline_strict" / "results" / BACKBONE_KEY
OUT_DIR.mkdir(parents=True, exist_ok=True)

# save preds
pd.DataFrame({"clip_id": val_ids,  **{f"pred_{c}": pred_val[:,i]  for i,c in enumerate(label_cols)}})\
  .to_csv(OUT_DIR / "pred_val.csv", index=False)
pd.DataFrame({"clip_id": test_ids, **{f"pred_{c}": pred_test[:,i] for i,c in enumerate(label_cols)}})\
  .to_csv(OUT_DIR / "pred_test.csv", index=False)

# save metrics
m_val.to_csv(OUT_DIR / "metrics_val.csv", index=False)
m_test.to_csv(OUT_DIR / "metrics_test.csv", index=False)

# save scaler + model
joblib.dump(scaler, OUT_DIR / "scaler.joblib")
joblib.dump(model,  OUT_DIR / "ridge_multioutput.joblib")

# save meta
meta = {
    "backbone": BACKBONE_KEY,
    "alpha": float(best_alpha),
    "label_cols": label_cols,
    "shapes": {"train": list(X_train.shape), "val": list(X_val.shape), "test": list(X_test.shape)},
}
import json
(Path(OUT_DIR) / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("\nSaved to:", OUT_DIR)


X_train: (5936, 768) | X_val: (1999, 768) | X_test: (2039, 768)
Label cols: ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
alpha=0.1    | val mean MAE=0.108051
alpha=1.0    | val mean MAE=0.107596
alpha=10.0   | val mean MAE=0.105656
alpha=100.0  | val mean MAE=0.102952

Best alpha = 100.0 (val mean MAE = 0.102952)

== wav2vec2 | STRICT | VAL ==


,trait,Acc(1-MAE),MAE,RMSE,R2
0,extraversion,0.895887,0.104113,0.130765,0.230800
1,neuroticism,0.896922,0.103078,0.130241,0.247400
2,agreeableness,0.900710,0.099290,0.125589,0.092921
3,conscientiousness,0.893812,0.106188,0.133761,0.219859
4,openness,0.897906,0.102094,0.128266,0.207923
mean,mean,0.897048,0.102952,0.129724,0.199780



== wav2vec2 | STRICT | TEST ==


,trait,Acc(1-MAE),MAE,RMSE,R2
0,extraversion,0.896872,0.103128,0.129611,0.295224
1,neuroticism,0.895380,0.104620,0.132294,0.310689
2,agreeableness,0.897037,0.102963,0.129157,0.127970
3,conscientiousness,0.893041,0.106959,0.135585,0.273979
4,openness,0.899249,0.100751,0.126105,0.283739
mean,mean,0.896316,0.103684,0.130551,0.258320



Saved to: E:\tugas-akhir-qiqi\output\baseline_strict\results\wav2vec2


# **REUSE EMBEDDING AND ALIGNING TO STRICT - WAVLM**

In [18]:
# =========================
# 2) Reuse embedding dari baseline_official -> align ke STRICT split (by clip_id)
# =========================

import numpy as np
import pandas as pd
from pathlib import Path

assert "ROOT" in globals()
assert "df_train" in globals() and "df_val" in globals() and "df_test" in globals()

BACKBONE_KEY = "wavlm"   # ganti: "wav2vec2" / "wavlm"

# lokasi embedding yang sudah ada (official)
EMB_DIR_OFF = ROOT / "output" / "baseline_official" / "embeddings" / BACKBONE_KEY
assert EMB_DIR_OFF.exists(), f"Folder embedding official tidak ada: {EMB_DIR_OFF}"

def load_official_split(split: str):
    X = np.load(EMB_DIR_OFF / f"{split}_emb.npy", mmap_mode="r")
    ids = pd.read_csv(EMB_DIR_OFF / f"{split}_clip_id.csv")["clip_id"].astype(str).tolist()
    return X, ids

# load pool embedding dari official train/val/test
X_off = {}
ids_off = {}

for split in ["train", "val", "test"]:
    X_off[split], ids_off[split] = load_official_split(split)
    print(f"[OFFICIAL] {split}: X={X_off[split].shape} ids={len(ids_off[split])}")

# build mapping clip_id -> (split, idx)
id2loc = {}
dup = 0
for split in ["train", "val", "test"]:
    for i, cid in enumerate(ids_off[split]):
        if cid in id2loc:
            dup += 1
        id2loc[cid] = (split, i)

if dup > 0:
    print(f"[WARN] Duplicated clip_id across official splits: {dup} (harusnya 0).")

H = int(X_off["train"].shape[1])
print("Hidden size (H):", H)

def build_X_from_strict(df_split: pd.DataFrame, name: str):
    ids = df_split["clip_id"].astype(str).tolist()
    X = np.zeros((len(ids), H), dtype=np.float32)

    missing = []
    for j, cid in enumerate(ids):
        loc = id2loc.get(cid)
        if loc is None:
            missing.append(cid)
            continue
        sp, i = loc
        X[j] = X_off[sp][i]  # ambil dari memmap official

    if missing:
        print(f"[WARN] {name}: missing embeddings = {len(missing)} / {len(ids)} (contoh 5): {missing[:5]}")
        # kalau kamu mau strict fail biar ketahuan:
        # raise ValueError(f"Missing embeddings in {name}: {len(missing)}")

    print(f"[STRICT] {name}: X={X.shape} ids={len(ids)}")
    return X, ids

# align ke strict split
X_train, train_ids = build_X_from_strict(df_train, "train")
X_val,   val_ids   = build_X_from_strict(df_val,   "val")
X_test,  test_ids  = build_X_from_strict(df_test,  "test")

# (opsional) cache ulang supaya notebook strict mirip baseline official
STRICT_EMB_DIR = ROOT / "output" / "baseline_strict" / "embeddings" / BACKBONE_KEY
STRICT_EMB_DIR.mkdir(parents=True, exist_ok=True)

np.save(STRICT_EMB_DIR / "train_emb.npy", X_train)
np.save(STRICT_EMB_DIR / "val_emb.npy",   X_val)
np.save(STRICT_EMB_DIR / "test_emb.npy",  X_test)

pd.DataFrame({"clip_id": train_ids}).to_csv(STRICT_EMB_DIR / "train_clip_id.csv", index=False)
pd.DataFrame({"clip_id": val_ids}).to_csv(STRICT_EMB_DIR / "val_clip_id.csv", index=False)
pd.DataFrame({"clip_id": test_ids}).to_csv(STRICT_EMB_DIR / "test_clip_id.csv", index=False)

print("Saved strict-aligned embeddings to:", STRICT_EMB_DIR)


[OFFICIAL] train: X=(5988, 768) ids=5988
[OFFICIAL] val: X=(1994, 768) ids=1994
[OFFICIAL] test: X=(1992, 768) ids=1992
Hidden size (H): 768
[STRICT] train: X=(5936, 768) ids=5936
[STRICT] val: X=(1999, 768) ids=1999
[STRICT] test: X=(2039, 768) ids=2039
Saved strict-aligned embeddings to: E:\tugas-akhir-qiqi\output\baseline_strict\embeddings\wavlm


In [19]:
# =========================
# 3) Train Ridge Regressor (baseline) + Eval (STRICT)
# =========================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# --- pilih backbone yang embedding-nya kamu reuse
BACKBONE_KEY = "wavlm"  # "wav2vec2" / "wavlm"

assert "ROOT" in globals()
assert "df_train" in globals() and "df_val" in globals() and "df_test" in globals()

EMB_DIR = ROOT / "output" / "baseline_strict" / "embeddings" / BACKBONE_KEY
assert EMB_DIR.exists(), f"EMB_DIR strict tidak ada: {EMB_DIR}"

def load_split(split: str):
    X = np.load(EMB_DIR / f"{split}_emb.npy")
    ids = pd.read_csv(EMB_DIR / f"{split}_clip_id.csv")["clip_id"].astype(str).tolist()
    return X, ids

def align_X_to_df(X, ids, df, name=""):
    id2i = {cid: i for i, cid in enumerate(ids)}
    target_ids = df["clip_id"].astype(str).tolist()

    order = []
    missing = []
    for cid in target_ids:
        i = id2i.get(cid)
        if i is None:
            missing.append(cid)
            order.append(-1)
        else:
            order.append(i)

    if missing:
        print(f"[WARN] {name}: missing embeddings = {len(missing)} / {len(target_ids)} (contoh 5): {missing[:5]}")
        # kalau mau strict fail:
        raise ValueError(f"Missing embeddings for {name}: {len(missing)}")

    X_aligned = X[np.array(order)]
    return X_aligned, target_ids

# --- load + align to strict dfs (important)
X_train_raw, train_ids_raw = load_split("train")
X_val_raw,   val_ids_raw   = load_split("val")
X_test_raw,  test_ids_raw  = load_split("test")

X_train, train_ids = align_X_to_df(X_train_raw, train_ids_raw, df_train, "train")
X_val,   val_ids   = align_X_to_df(X_val_raw,   val_ids_raw,   df_val,   "val")
X_test,  test_ids  = align_X_to_df(X_test_raw,  test_ids_raw,  df_test,  "test")

print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

# --- auto-detect label columns (ambil 5 kolom numerik yang bukan id/path/group/split)
def detect_label_cols(df):
    exclude = set([c for c in df.columns if any(k in c.lower() for k in ["clip", "id", "path", "file", "group", "split", "ethnicity", "gender", "avg_trait"])])
    num_cols = [c for c in df.columns if (c not in exclude) and pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) < 5:
        raise ValueError(f"Kolom numerik kandidat label kurang dari 5: {num_cols}")
    return num_cols[:5]

label_cols = detect_label_cols(df_train)
print("Label cols:", label_cols)

y_train = df_train[label_cols].to_numpy(dtype=np.float32)
y_val   = df_val[label_cols].to_numpy(dtype=np.float32)
y_test  = df_test[label_cols].to_numpy(dtype=np.float32)

# --- scale embeddings (fit hanya di train)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

def metrics(y_true, y_pred, name=""):
    mae  = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))
    r2   = r2_score(y_true, y_pred, multioutput="raw_values")

    acc = 1.0 - mae
    mean_acc = acc.mean()

    dfm = pd.DataFrame({
        "trait": label_cols,
        "Acc(1-MAE)": acc,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })
    dfm.loc["mean"] = ["mean", mean_acc, mae.mean(), rmse.mean(), r2.mean()]

    print(f"\n== {name} ==")
    try:
        display(dfm)
    except Exception:
        print(dfm)
    return dfm

# --- (opsional) tuning alpha pakai mean MAE val
ALPHAS = [0.1, 1.0, 10.0, 100.0]
best_alpha = None
best_val = 1e9
best_model = None

for a in ALPHAS:
    ridge = Ridge(alpha=a, random_state=42)
    model = MultiOutputRegressor(ridge)
    model.fit(X_train_s, y_train)
    pred_val = model.predict(X_val_s)
    mae_val = mean_absolute_error(y_val, pred_val)  # mean MAE (gabungan 5 trait)
    print(f"alpha={a:<6} | val mean MAE={mae_val:.6f}")
    if mae_val < best_val:
        best_val = mae_val
        best_alpha = a
        best_model = model

print(f"\nBest alpha = {best_alpha} (val mean MAE = {best_val:.6f})")

# --- final predict (pakai best alpha)
model = best_model
pred_val  = model.predict(X_val_s)
pred_test = model.predict(X_test_s)

m_val  = metrics(y_val,  pred_val,  f"{BACKBONE_KEY} | STRICT | VAL")
m_test = metrics(y_test, pred_test, f"{BACKBONE_KEY} | STRICT | TEST")

# --- save outputs
OUT_DIR = ROOT / "output" / "baseline_strict" / "results" / BACKBONE_KEY
OUT_DIR.mkdir(parents=True, exist_ok=True)

# save preds
pd.DataFrame({"clip_id": val_ids,  **{f"pred_{c}": pred_val[:,i]  for i,c in enumerate(label_cols)}})\
  .to_csv(OUT_DIR / "pred_val.csv", index=False)
pd.DataFrame({"clip_id": test_ids, **{f"pred_{c}": pred_test[:,i] for i,c in enumerate(label_cols)}})\
  .to_csv(OUT_DIR / "pred_test.csv", index=False)

# save metrics
m_val.to_csv(OUT_DIR / "metrics_val.csv", index=False)
m_test.to_csv(OUT_DIR / "metrics_test.csv", index=False)

# save scaler + model
joblib.dump(scaler, OUT_DIR / "scaler.joblib")
joblib.dump(model,  OUT_DIR / "ridge_multioutput.joblib")

# save meta
meta = {
    "backbone": BACKBONE_KEY,
    "alpha": float(best_alpha),
    "label_cols": label_cols,
    "shapes": {"train": list(X_train.shape), "val": list(X_val.shape), "test": list(X_test.shape)},
}
import json
(Path(OUT_DIR) / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("\nSaved to:", OUT_DIR)


X_train: (5936, 768) | X_val: (1999, 768) | X_test: (2039, 768)
Label cols: ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
alpha=0.1    | val mean MAE=0.101763
alpha=1.0    | val mean MAE=0.101632
alpha=10.0   | val mean MAE=0.100778
alpha=100.0  | val mean MAE=0.098945

Best alpha = 100.0 (val mean MAE = 0.098945)

== wavlm | STRICT | VAL ==


,trait,Acc(1-MAE),MAE,RMSE,R2
0,extraversion,0.900763,0.099237,0.123809,0.310465
1,neuroticism,0.900619,0.099381,0.125107,0.305563
2,agreeableness,0.903744,0.096256,0.121316,0.153592
3,conscientiousness,0.900028,0.099972,0.127012,0.296597
4,openness,0.900120,0.099880,0.124459,0.254250
mean,mean,0.901055,0.098945,0.124341,0.264093



== wavlm | STRICT | TEST ==


,trait,Acc(1-MAE),MAE,RMSE,R2
0,extraversion,0.899133,0.100867,0.126808,0.325372
1,neuroticism,0.896522,0.103478,0.131382,0.320160
2,agreeableness,0.900072,0.099928,0.126192,0.167554
3,conscientiousness,0.895737,0.104263,0.132144,0.310369
4,openness,0.902192,0.097808,0.123330,0.314922
mean,mean,0.898731,0.101269,0.127971,0.287675



Saved to: E:\tugas-akhir-qiqi\output\baseline_strict\results\wavlm


# **PERBANDINGAN BACKBONE TRANSFORMER STRICT**

In [20]:
import pandas as pd
from pathlib import Path

RESULT_ROOT = ROOT / "output" / "baseline_strict" / "results"

METHODS = ["wav2vec2", "hubert", "wavlm"]  # <- tambahin "egemaps" kalau ada di strict juga

def load_metrics(method: str, split: str) -> pd.DataFrame:
    p = RESULT_ROOT / method / f"metrics_{split}.csv"
    if not p.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {p}")
    df = pd.read_csv(p)
    df["trait"] = df["trait"].astype(str)
    return df

def summarize_mean(df: pd.DataFrame) -> dict:
    mean_row = df[df["trait"] == "mean"]
    if len(mean_row) > 0:
        r = mean_row.iloc[0]
        return {
            "Acc(1-MAE)": r.get("Acc(1-MAE)", None),
            "MAE": r.get("MAE", None),
            "RMSE": r.get("RMSE", None),
            "R2": r.get("R2", None),
        }
    df2 = df[df["trait"] != "mean"].copy()
    return {
        "Acc(1-MAE)": df2["Acc(1-MAE)"].mean() if "Acc(1-MAE)" in df2.columns else None,
        "MAE": df2["MAE"].mean(),
        "RMSE": df2["RMSE"].mean(),
        "R2": df2["R2"].mean(),
    }

def add_ranks(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "Acc(1-MAE)" in out.columns and out["Acc(1-MAE)"].notna().any():
        out["rank_Acc"] = out["Acc(1-MAE)"].rank(ascending=False, method="min")
    out["rank_MAE"] = out["MAE"].rank(ascending=True, method="min")
    out["rank_RMSE"] = out["RMSE"].rank(ascending=True, method="min")
    out["rank_R2"] = out["R2"].rank(ascending=False, method="min")

    rank_cols = [c for c in ["rank_Acc", "rank_MAE", "rank_RMSE", "rank_R2"] if c in out.columns]
    out["rank_total"] = out[rank_cols].mean(axis=1)
    return out.sort_values("rank_total")

# --- build mean tables
rows_val, rows_test = [], []
for m in METHODS:
    dfv = load_metrics(m, "val")
    dft = load_metrics(m, "test")

    rows_val.append({"method": m, **summarize_mean(dfv)})
    rows_test.append({"method": m, **summarize_mean(dft)})

cmp_val = add_ranks(pd.DataFrame(rows_val))
cmp_test = add_ranks(pd.DataFrame(rows_test))

print("=== MEAN METRICS (STRICT | VAL) ===")
display(cmp_val)

print("\n=== MEAN METRICS (STRICT | TEST) ===")
display(cmp_test)

# --- optional: per-trait table (VAL/TEST) for MAE/Acc/R2
def per_trait_table(split="val", metric_col="MAE"):
    tables = []
    for m in METHODS:
        df = load_metrics(m, split)
        df = df[df["trait"] != "mean"][["trait", metric_col]].copy()
        df = df.rename(columns={metric_col: m})
        tables.append(df.set_index("trait"))
    return pd.concat(tables, axis=1)

print("\n=== Per-trait Acc(1-MAE) (STRICT | VAL) ===")
display(per_trait_table("val", "Acc(1-MAE)"))

print("\n=== Per-trait MAE (STRICT | VAL) ===")
display(per_trait_table("val", "MAE"))

print("\n=== Per-trait R2 (STRICT | VAL) ===")
display(per_trait_table("val", "R2"))

print("\n=== Per-trait MAE (STRICT | TEST) ===")
display(per_trait_table("test", "MAE"))


=== MEAN METRICS (STRICT | VAL) ===


,method,Acc(1-MAE),MAE,RMSE,R2,rank_Acc,rank_MAE,rank_RMSE,rank_R2,rank_total
2,wavlm,0.901055,0.098945,0.124341,0.264093,1.0,1.0,1.0,1.0,1.0
1,hubert,0.898972,0.101027,0.127312,0.228886,2.0,2.0,2.0,2.0,2.0
0,wav2vec2,0.897047,0.102952,0.129724,0.199780,3.0,3.0,3.0,3.0,3.0



=== MEAN METRICS (STRICT | TEST) ===


,method,Acc(1-MAE),MAE,RMSE,R2,rank_Acc,rank_MAE,rank_RMSE,rank_R2,rank_total
2,wavlm,0.898731,0.101269,0.127971,0.287675,1.0,1.0,1.0,1.0,1.0
1,hubert,0.896639,0.103362,0.130347,0.261567,2.0,2.0,2.0,2.0,2.0
0,wav2vec2,0.896316,0.103684,0.130551,0.258320,3.0,3.0,3.0,3.0,3.0



=== Per-trait Acc(1-MAE) (STRICT | VAL) ===


...

,wav2vec2,hubert,wavlm
trait,,,
extraversion,0.104113,0.101508,0.099237
neuroticism,0.103078,0.102425,0.099381
agreeableness,0.099290,0.097976,0.096256
conscientiousness,0.106188,0.103584,0.099972
openness,0.102094,0.099644,0.099880



=== Per-trait R2 (STRICT | VAL) ===


,wav2vec2,hubert,wavlm
trait,,,
extraversion,0.230800,0.277348,0.310465
neuroticism,0.247400,0.262564,0.305563
agreeableness,0.092921,0.114129,0.153592
conscientiousness,0.219859,0.246699,0.296597
openness,0.207923,0.243692,0.254250



=== Per-trait MAE (STRICT | TEST) ===


,wav2vec2,hubert,wavlm
trait,,,
extraversion,0.103128,0.103305,0.100867
neuroticism,0.104620,0.105762,0.103478
agreeableness,0.102963,0.100425,0.099928
conscientiousness,0.106959,0.106143,0.104263
openness,0.100751,0.101173,0.097808


# **REUSE EMBEDDING AND ALIGNING TO STRICT - EGEMAPS**

In [21]:
# =========================
# Reuse eGeMAPS (OFFICIAL) -> align to STRICT split (by clip_id) + save
# =========================

import pandas as pd
from pathlib import Path

assert "ROOT" in globals()
assert "df_train" in globals() and "df_val" in globals() and "df_test" in globals()

EG_OFF_DIR = ROOT / "output" / "baseline_official" / "egemaps"
assert EG_OFF_DIR.exists(), f"EG_OFF_DIR tidak ada: {EG_OFF_DIR}"

# load official egemaps (train/val/test) lalu gabung jadi pool
eg_train_off = pd.read_csv(EG_OFF_DIR / "train_egemaps.csv")
eg_val_off   = pd.read_csv(EG_OFF_DIR / "val_egemaps.csv")
eg_test_off  = pd.read_csv(EG_OFF_DIR / "test_egemaps.csv")

for df in [eg_train_off, eg_val_off, eg_test_off]:
    df["clip_id"] = df["clip_id"].astype(str)

eg_pool = pd.concat([eg_train_off, eg_val_off, eg_test_off], ignore_index=True)

# kalau ada duplikat clip_id, keep yang pertama (harusnya tidak, tapi aman)
eg_pool = eg_pool.drop_duplicates(subset=["clip_id"], keep="first").set_index("clip_id")

print("eg_pool:", eg_pool.shape, "| unique clip_id:", eg_pool.index.nunique())

def align_egemaps(df_split: pd.DataFrame, name: str) -> pd.DataFrame:
    ids = df_split["clip_id"].astype(str).tolist()
    missing = [cid for cid in ids if cid not in eg_pool.index]
    if missing:
        print(f"[WARN] {name}: missing egemaps = {len(missing)} / {len(ids)} (contoh 5): {missing[:5]}")
        raise ValueError(f"Missing eGeMAPS for {name}: {len(missing)}")
    out = eg_pool.loc[ids].reset_index()  # clip_id balik jadi kolom
    print(f"[STRICT] {name}: {out.shape}")
    return out

eg_train_strict = align_egemaps(df_train, "train")
eg_val_strict   = align_egemaps(df_val,   "val")
eg_test_strict  = align_egemaps(df_test,  "test")

# save ke baseline_strict/egemaps (biar struktur sama seperti official)
EG_STRICT_DIR = ROOT / "output" / "baseline_strict" / "egemaps"
EG_STRICT_DIR.mkdir(parents=True, exist_ok=True)

eg_train_strict.to_csv(EG_STRICT_DIR / "train_egemaps.csv", index=False)
eg_val_strict.to_csv(EG_STRICT_DIR / "val_egemaps.csv", index=False)
eg_test_strict.to_csv(EG_STRICT_DIR / "test_egemaps.csv", index=False)

print("Saved strict eGeMAPS to:", EG_STRICT_DIR)


eg_pool: (9974, 88) | unique clip_id: 9974
[STRICT] train: (5936, 89)
[STRICT] val: (1999, 89)
[STRICT] test: (2039, 89)
Saved strict eGeMAPS to: E:\tugas-akhir-qiqi\output\baseline_strict\egemaps


In [22]:
# =========================
# eGeMAPS (STRICT) -> StandardScaler + Ridge + Tuning alpha + Eval + Save
# =========================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib, json

assert "ROOT" in globals()
assert "df_train" in globals() and "df_val" in globals() and "df_test" in globals()

EG_DIR = ROOT / "output" / "baseline_strict" / "egemaps"
assert EG_DIR.exists(), f"EG_DIR strict tidak ada: {EG_DIR}"

# --- load eGeMAPS features (strict)
eg_train = pd.read_csv(EG_DIR / "train_egemaps.csv")
eg_val   = pd.read_csv(EG_DIR / "val_egemaps.csv")
eg_test  = pd.read_csv(EG_DIR / "test_egemaps.csv")

print("eGeMAPS strict train/val/test:", eg_train.shape, eg_val.shape, eg_test.shape)

# --- pastikan urutan sama dengan df_train/df_val/df_test (berdasarkan clip_id)
eg_train["clip_id"] = eg_train["clip_id"].astype(str)
eg_val["clip_id"]   = eg_val["clip_id"].astype(str)
eg_test["clip_id"]  = eg_test["clip_id"].astype(str)

eg_train = eg_train.set_index("clip_id").loc[df_train["clip_id"].astype(str)].reset_index()
eg_val   = eg_val.set_index("clip_id").loc[df_val["clip_id"].astype(str)].reset_index()
eg_test  = eg_test.set_index("clip_id").loc[df_test["clip_id"].astype(str)].reset_index()

# --- X (fitur) dan ids
train_ids = eg_train["clip_id"].astype(str).tolist()
val_ids   = eg_val["clip_id"].astype(str).tolist()
test_ids  = eg_test["clip_id"].astype(str).tolist()

X_train = eg_train.drop(columns=["clip_id"]).to_numpy(dtype=np.float32)
X_val   = eg_val.drop(columns=["clip_id"]).to_numpy(dtype=np.float32)
X_test  = eg_test.drop(columns=["clip_id"]).to_numpy(dtype=np.float32)

print("X shapes:", X_train.shape, X_val.shape, X_test.shape)

# --- detect label cols (5 trait) dari metadata strict
def detect_label_cols(df):
    exclude = set([c for c in df.columns if any(k in c.lower() for k in ["clip", "id", "path", "file", "group", "split", "ethnicity", "gender", "avg_trait"])])
    num_cols = [c for c in df.columns if (c not in exclude) and pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) < 5:
        raise ValueError(f"Kolom numerik kandidat label kurang dari 5: {num_cols}")
    return num_cols[:5]

label_cols = detect_label_cols(df_train)
print("Label cols:", label_cols)

y_train = df_train[label_cols].to_numpy(dtype=np.float32)
y_val   = df_val[label_cols].to_numpy(dtype=np.float32)
y_test  = df_test[label_cols].to_numpy(dtype=np.float32)

# --- scale (fit hanya di train)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# --- tuning alpha on VAL (mean MAE gabungan 5 trait)
ALPHAS = [0.1, 1.0, 10.0, 100.0]  # boleh kamu perluas
best_alpha = None
best_val_mae = 1e9
best_model = None

for a in ALPHAS:
    ridge = Ridge(alpha=a, random_state=42)
    model = MultiOutputRegressor(ridge)
    model.fit(X_train_s, y_train)

    pred_val_tmp = model.predict(X_val_s)
    mae_val_tmp = mean_absolute_error(y_val, pred_val_tmp)  # mean MAE
    print(f"alpha={a:<6} | val mean MAE={mae_val_tmp:.6f}")

    if mae_val_tmp < best_val_mae:
        best_val_mae = mae_val_tmp
        best_alpha = a
        best_model = model

print(f"\nBest alpha = {best_alpha} (val mean MAE = {best_val_mae:.6f})")

# --- final predict pakai best_model
model = best_model
pred_val  = model.predict(X_val_s)
pred_test = model.predict(X_test_s)

# --- metrics (Acc = 1 - MAE)
def metrics(y_true, y_pred, name=""):
    mae  = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))
    r2   = r2_score(y_true, y_pred, multioutput="raw_values")
    acc  = 1.0 - mae

    dfm = pd.DataFrame({
        "trait": label_cols,
        "Acc(1-MAE)": acc,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })
    dfm.loc["mean"] = ["mean", acc.mean(), mae.mean(), rmse.mean(), r2.mean()]
    print(f"\n== {name} ==")
    display(dfm)
    return dfm

m_val  = metrics(y_val,  pred_val,  f"eGeMAPS | STRICT | VAL | alpha={best_alpha}")
m_test = metrics(y_test, pred_test, f"eGeMAPS | STRICT | TEST | alpha={best_alpha}")

# --- save outputs
OUT_DIR = ROOT / "output" / "baseline_strict" / "results" / "egemaps"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({"clip_id": val_ids, **{f"pred_{c}": pred_val[:,i] for i,c in enumerate(label_cols)}})\
  .to_csv(OUT_DIR / "pred_val.csv", index=False)
pd.DataFrame({"clip_id": test_ids, **{f"pred_{c}": pred_test[:,i] for i,c in enumerate(label_cols)}})\
  .to_csv(OUT_DIR / "pred_test.csv", index=False)

m_val.to_csv(OUT_DIR / "metrics_val.csv", index=False)
m_test.to_csv(OUT_DIR / "metrics_test.csv", index=False)

joblib.dump(scaler, OUT_DIR / "scaler.joblib")
joblib.dump(model,  OUT_DIR / "ridge_multioutput.joblib")

meta = {
    "method": "egemaps",
    "split": "strict",
    "alpha": float(best_alpha) if best_alpha is not None else None,
    "alphas_tried": ALPHAS,
    "label_cols": label_cols,
    "x_shapes": {"train": list(X_train.shape), "val": list(X_val.shape), "test": list(X_test.shape)},
}
(Path(OUT_DIR) / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("\nSaved to:", OUT_DIR)


eGeMAPS strict train/val/test: (5936, 89) (1999, 89) (2039, 89)
X shapes: (5936, 88) (1999, 88) (2039, 88)
Label cols: ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
alpha=0.1    | val mean MAE=0.106880
alpha=1.0    | val mean MAE=0.106783
alpha=10.0   | val mean MAE=0.106595
alpha=100.0  | val mean MAE=0.106109

Best alpha = 100.0 (val mean MAE = 0.106109)

== eGeMAPS | STRICT | VAL | alpha=100.0 ==


,trait,Acc(1-MAE),MAE,RMSE,R2
0,extraversion,0.891466,0.108534,0.135737,0.171199
1,neuroticism,0.892691,0.107309,0.135894,0.180651
2,agreeableness,0.901525,0.098475,0.126090,0.085664
3,conscientiousness,0.887385,0.112615,0.140701,0.136802
4,openness,0.896387,0.103613,0.130855,0.175620
mean,mean,0.893891,0.106109,0.133856,0.149987



== eGeMAPS | STRICT | TEST | alpha=100.0 ==


,trait,Acc(1-MAE),MAE,RMSE,R2
0,extraversion,0.893571,0.106429,0.133046,0.257377
1,neuroticism,0.891371,0.108629,0.136633,0.264734
2,agreeableness,0.898105,0.101895,0.128243,0.140273
3,conscientiousness,0.885727,0.114273,0.142557,0.197393
4,openness,0.898689,0.101311,0.128125,0.260615
mean,mean,0.893493,0.106507,0.133721,0.224078



Saved to: E:\tugas-akhir-qiqi\output\baseline_strict\results\egemaps


# **PERBANDINGAN TOTAL**

In [23]:
import pandas as pd
from pathlib import Path

RESULT_ROOT = ROOT / "output" / "baseline_strict" / "results"

METHODS = ["wav2vec2", "hubert", "wavlm", "egemaps"]

def load_metrics(method: str, split: str) -> pd.DataFrame:
    p = RESULT_ROOT / method / f"metrics_{split}.csv"
    if not p.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {p}")
    df = pd.read_csv(p)
    df["trait"] = df["trait"].astype(str)
    return df

def summarize_mean(df: pd.DataFrame) -> dict:
    mean_row = df[df["trait"] == "mean"]
    if len(mean_row) > 0:
        r = mean_row.iloc[0]
        return {
            "Acc(1-MAE)": r.get("Acc(1-MAE)", None),
            "MAE": r.get("MAE", None),
            "RMSE": r.get("RMSE", None),
            "R2": r.get("R2", None),
        }
    df2 = df[df["trait"] != "mean"].copy()
    return {
        "Acc(1-MAE)": df2["Acc(1-MAE)"].mean() if "Acc(1-MAE)" in df2.columns else None,
        "MAE": df2["MAE"].mean(),
        "RMSE": df2["RMSE"].mean(),
        "R2": df2["R2"].mean(),
    }

def add_ranks(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "Acc(1-MAE)" in out.columns and out["Acc(1-MAE)"].notna().any():
        out["rank_Acc"] = out["Acc(1-MAE)"].rank(ascending=False, method="min")
    out["rank_MAE"] = out["MAE"].rank(ascending=True, method="min")
    out["rank_RMSE"] = out["RMSE"].rank(ascending=True, method="min")
    out["rank_R2"] = out["R2"].rank(ascending=False, method="min")

    rank_cols = [c for c in ["rank_Acc", "rank_MAE", "rank_RMSE", "rank_R2"] if c in out.columns]
    out["rank_total"] = out[rank_cols].mean(axis=1)
    return out.sort_values("rank_total")

# --- build mean tables
rows_val, rows_test = [], []
for m in METHODS:
    dfv = load_metrics(m, "val")
    dft = load_metrics(m, "test")
    rows_val.append({"method": m, **summarize_mean(dfv)})
    rows_test.append({"method": m, **summarize_mean(dft)})

cmp_val = add_ranks(pd.DataFrame(rows_val))
cmp_test = add_ranks(pd.DataFrame(rows_test))

print("=== MEAN METRICS (STRICT | VAL) ===")
display(cmp_val)

print("\n=== MEAN METRICS (STRICT | TEST) ===")
display(cmp_test)

# --- optional: per-trait table (VAL) for MAE/Acc/R2 (seperti official)
def per_trait_table(split="val", metric_col="MAE"):
    tables = []
    for m in METHODS:
        df = load_metrics(m, split)
        df = df[df["trait"] != "mean"][["trait", metric_col]].copy()
        df = df.rename(columns={metric_col: m})
        tables.append(df.set_index("trait"))
    return pd.concat(tables, axis=1)

print("\n=== Per-trait Acc(1-MAE) (STRICT | VAL) ===")
display(per_trait_table("val", "Acc(1-MAE)"))

print("\n=== Per-trait MAE (STRICT | VAL) ===")
display(per_trait_table("val", "MAE"))

print("\n=== Per-trait R2 (STRICT | VAL) ===")
display(per_trait_table("val", "R2"))

print("\n=== Per-trait MAE (STRICT | TEST) ===")
display(per_trait_table("test", "MAE"))


=== MEAN METRICS (STRICT | VAL) ===


,method,Acc(1-MAE),MAE,RMSE,R2,rank_Acc,rank_MAE,rank_RMSE,rank_R2,rank_total
2,wavlm,0.901055,0.098945,0.124341,0.264093,1.0,1.0,1.0,1.0,1.0
1,hubert,0.898972,0.101027,0.127312,0.228886,2.0,2.0,2.0,2.0,2.0
0,wav2vec2,0.897047,0.102952,0.129724,0.199780,3.0,3.0,3.0,3.0,3.0
3,egemaps,0.893891,0.106109,0.133856,0.149987,4.0,4.0,4.0,4.0,4.0



=== MEAN METRICS (STRICT | TEST) ===


,method,Acc(1-MAE),MAE,RMSE,R2,rank_Acc,rank_MAE,rank_RMSE,rank_R2,rank_total
2,wavlm,0.898731,0.101269,0.127971,0.287675,1.0,1.0,1.0,1.0,1.0
1,hubert,0.896639,0.103362,0.130347,0.261567,2.0,2.0,2.0,2.0,2.0
0,wav2vec2,0.896316,0.103684,0.130551,0.258320,3.0,3.0,3.0,3.0,3.0
3,egemaps,0.893493,0.106507,0.133721,0.224078,4.0,4.0,4.0,4.0,4.0



=== Per-trait Acc(1-MAE) (STRICT | VAL) ===


...

,wav2vec2,hubert,wavlm,egemaps
trait,,,,
extraversion,0.104113,0.101508,0.099237,0.108534
neuroticism,0.103078,0.102425,0.099381,0.107309
agreeableness,0.099290,0.097976,0.096256,0.098475
conscientiousness,0.106188,0.103584,0.099972,0.112615
openness,0.102094,0.099644,0.099880,0.103613



=== Per-trait R2 (STRICT | VAL) ===


,wav2vec2,hubert,wavlm,egemaps
trait,,,,
extraversion,0.230800,0.277348,0.310465,0.171199
neuroticism,0.247400,0.262564,0.305563,0.180651
agreeableness,0.092921,0.114129,0.153592,0.085664
conscientiousness,0.219859,0.246699,0.296597,0.136802
openness,0.207923,0.243692,0.254250,0.175620



=== Per-trait MAE (STRICT | TEST) ===


,wav2vec2,hubert,wavlm,egemaps
trait,,,,
extraversion,0.103128,0.103305,0.100867,0.106429
neuroticism,0.104620,0.105762,0.103478,0.108629
agreeableness,0.102963,0.100425,0.099928,0.101895
conscientiousness,0.106959,0.106143,0.104263,0.114273
openness,0.100751,0.101173,0.097808,0.101311


# **BUKTI STRICT BERSIH**

In [24]:
import numpy as np

# 1) cek overlap clip_id antar split (HARUS 0)
tr = set(df_train["clip_id"].astype(str))
va = set(df_val["clip_id"].astype(str))
te = set(df_test["clip_id"].astype(str))

print("overlap train∩val:", len(tr & va))
print("overlap train∩test:", len(tr & te))
print("overlap val∩test:", len(va & te))
assert len(tr & va) == 0 and len(tr & te) == 0 and len(va & te) == 0

# 2) kalau ada group_id, cek leakage group (HARUS 0)
if "group_id" in df_train.columns:
    gtr = set(df_train["group_id"].astype(str))
    gva = set(df_val["group_id"].astype(str))
    gte = set(df_test["group_id"].astype(str))
    print("group overlap train∩val:", len(gtr & gva))
    print("group overlap train∩test:", len(gtr & gte))
    print("group overlap val∩test:", len(gva & gte))
    assert len(gtr & gva) == 0 and len(gtr & gte) == 0 and len(gva & gte) == 0

# 3) cek baris embedding yang nol total (indikasi missing)
def zero_rows(X):
    return int(np.sum(np.linalg.norm(X, axis=1) == 0))

print("zero rows train:", zero_rows(X_train))
print("zero rows val  :", zero_rows(X_val))
print("zero rows test :", zero_rows(X_test))
assert zero_rows(X_train) == 0 and zero_rows(X_val) == 0 and zero_rows(X_test) == 0

# 4) cek label_cols bener-bener 5 trait (manual check!)
print("label_cols:", label_cols)


overlap train∩val: 0
overlap train∩test: 0
overlap val∩test: 0
group overlap train∩val: 0
group overlap train∩test: 0
group overlap val∩test: 0
zero rows train: 0
zero rows val  : 0
zero rows test : 0
label_cols: ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
